# Learned thinking vs explicit search (A100)

Trains `expert/train_looped.py` on the self-play run's 128-simulation SEARCH targets (`selfplay/buffers_21x15.npz`) plus
puzzles up to 4 jumps: the 2-block looper (2 shared blocks x T loops), the 6-block looper (parameters matched to the baseline; one loop = one
fixed-network pass), and the fixed 6-layer baseline, none with search. The looped
run first measures the self-play network WITH Gumbel search (0/8/32/128 sims) on the same puzzles; every result is
logged with its compute in block-equivalents (search: ~6(N+1) blocks/move; looped: ~2T), so thinking and search sit on
one axis. Needs `phutball/data/expert_data/puzzle_eval_deep.npz`. Logs: `phutball/looped/{looped,fixed}/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/phutball'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install mctx==0.0.6 2>&1 | tail -1
!mkdir -p /content/data && rsync -a --exclude '*.tmp*' $DRIVE/data/expert_data /content/data/
import os; assert os.path.exists('/content/data/expert_data/puzzle_eval_deep.npz'), 'upload puzzle_eval_deep.npz to phutball/data/expert_data first'
TEACHER = f'{DRIVE}/selfplay/selfplay_21x15_final.pkl'; assert os.path.exists(TEACHER)
REPLAY = f'{DRIVE}/selfplay/buffers_21x15.npz'; assert os.path.exists(REPLAY)   # 128-sim search targets

In [ ]:
import subprocess, sys
RUNS = [   # (run folder, arguments): the 2-block looper (also measures the search baseline), the 6-block looper
           # (parameters matched to the baseline; one loop = one fixed-network pass), the fixed 6-layer baseline
    ('looped',  '--model looped --blocks 2 --loops-train 2,4,6,8 --eval-loops 1,2,4,8,16,32'),
    ('looped6', '--model looped --blocks 6 --loops-train 1,2,3 --eval-loops 1,2,3,4,6,8 --search-sims ""'),
    ('fixed',   '--model fixed'),
]
for name, extra in RUNS:
    run = f'{DRIVE}/looped/{name}'; os.makedirs(run, exist_ok=True)
    cmd = (f'{sys.executable} -u -m expert.train_looped {extra} --teacher {TEACHER} --replay {REPLAY} --run-dir {run} '
           f'--data-root /content/data --steps 6000 --batch 256 --eval-every 1000 --eval-per-depth 40')
    p = subprocess.Popen(cmd, shell=True, cwd='/content/phutball-jax', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    print(name, 'exit code', p.wait())